# BigQuery: Conversational Analytics Glossary Demo (February 2026 Preview)

This notebook demonstrates how to create **Custom Glossary Terms** for a BigQuery Conversational Analytics agent to help it correctly interpret business-specific terminology.

## Use Case
A retail company uses the term 'Active Customers' to mean users who have placed at least 3 orders in the last 12 months. Without a glossary, the AI might use a simpler definition. We'll define it correctly for the agent.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = bigquery.Client(project=project_id)

### 2. Define Custom Glossary Terms

We can define our terms in a JSON structure that the agent will use during the prompt interpretation phase.

In [ ]:
glossary_terms = [
    {
        "term": "Active Customer",
        "definition": "A user who has completed at least 3 orders in the last 12 months.",
        "sql_hint": "JOIN (SELECT user_id, COUNT(*) as order_count FROM orders WHERE status = 'Complete' AND created_at >= DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH) GROUP BY user_id) as sub ON users.id = sub.user_id WHERE order_count >= 3"
    },
    {
        "term": "Top Performance",
        "definition": "Refers to products with a sale price above $100 and a rating of 4.5+.",
        "sql_hint": "WHERE sale_price > 100 AND product_rating >= 4.5"
    }
]

print("Custom Glossary Terms defined:")
for entry in glossary_terms:
    print(f"- {entry['term']}: {entry['definition']}")

### 3. Test with a Conversational Prompt
When a user asks: "Show me the geographic distribution of Active Customers.", the agent now knows exactly which SQL logic to apply.

In [ ]:
prompt = "Show me the geographic distribution of Active Customers."
print(f"User Prompt: {prompt}")
print("Agent Step 1: Matching 'Active Customer' in Custom Glossary...")
print("Agent Step 2: Applying SQL Hint (JOIN with filtered orders table)...")
print("Agent Result: Correct SQL generated successfully.")

### 4. Things to remember or know
- **Precision at Scale**: Avoid AI hallucinations by explicitly defining 'Company Speak'.
- **Governance First**: Import existing terms from Dataplex Universal Catalog to ensure consistency across the enterprise.
- **Faster Insights**: Reduces the need for users to 're-prompt' the agent when it gets a definition wrong.